In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
import anndata as ad
from sklearn.decomposition import PCA

In [2]:
#with open("PRESAGE/cache/pathway_embeddings/k562_essen_prior.pkl", 'rb') as f:
#    l = pickle.load(f)

In [3]:
#l.shape

In [4]:
#l.head()

In [5]:
#l.index.name == None

In [6]:
adata = sc.read_h5ad('large_screens_per_ct/HCT.h5ad')
adata

AnnData object with n_obs × n_vars = 172801 × 7615
    obs: 'sample', 'num_features', 'guide_target', 'gene_target', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'pass_guide_filter', 'context', 'perturbation', 'perturbation_group', 'cell_type', 'dataset_source', 'lane_id', 'top_guide_UMI_counts', 'guide_id', 'perturbed_gene_id', 'guide_type', 'PuroR', 'guide_group', 'low_quality', 'gem_group', 'gene_id', 'transcript', 'gene_transcript', 'sgID_AB', 'mitopercent', 'UMI_count', 'z_gemgroup_UMI', 'core_scale_factor', 'core_adjusted_UMI_count', 'K562_TF_10_UF_10_rs_1_random', 'K562_TF_10_UF_10_rs_1_stratified', 'K562_TF_10_UF_10_rs_2_random', 'K562_TF_10_UF_10_rs_2_stratified', 'K562_TF_10_UF_10_rs_3_random', 'K562_TF_10_UF_10_rs_3_stratified', 'K562_TF_30_UF_10_rs_1_random', 'K562_TF_30_UF_10_rs_1_stratified', 'K562_TF_30_UF_10_rs_2_random', 'K562_TF_30_UF_10_rs_2_stratified', 'K562_TF_30_UF_10_rs_3_random', 'K562_TF_30_UF_10_rs_3_stratified', 'K562_TF_50_UF_10_

In [7]:
def compute_pseudobulk(
    adata: ad.AnnData, condition_field: str = "perturbation"
) -> pd.DataFrame:

    return pd.DataFrame(
        adata.X.toarray(), index=adata.obs[condition_field].tolist(), columns=adata.var_names
    ).pipe(lambda df: df.groupby(df.index).mean())

In [8]:
perturb_field = 'perturbation'
control_key = 'non-targeting'

In [9]:
adata.obs[perturb_field].value_counts()

perturbation
non-targeting    55259
FAM217B            200
MYRF               174
PIK3CG             171
CASP3              166
                 ...  
AKR7A2              11
ALG3                11
ZNF480              11
REV3L               11
ZNF708              11
Name: count, Length: 3068, dtype: int64

In [10]:
ctrl_idx = adata.obs[perturb_field] == control_key
controls = adata[ctrl_idx]

perturbs = adata[~ctrl_idx]

control_mean = np.mean(controls.X, axis=0) #, keepdims=True)
print("Computing pseudobulk...")
pseudobulk = compute_pseudobulk(perturbs, perturb_field)

X = pseudobulk.values
X -= control_mean
X = X.astype(np.float32)

Computing pseudobulk...


In [11]:
X

array([[-0.02277446, -0.17145205,  0.51165867, ..., -0.460747  ,
         0.09554577,  0.        ],
       [-0.01156116,  0.00919116, -0.16199684, ...,  0.27930033,
        -0.5089375 ,  0.        ],
       [ 0.03526187, -0.01786935, -0.11665964, ..., -0.18842447,
         0.27245724,  0.        ],
       ...,
       [ 0.10366416, -0.06074077,  0.11752832, ...,  0.17629874,
         0.06908941,  0.        ],
       [-0.34795213, -0.05310255, -0.09484243, ...,  0.1625731 ,
         0.3502773 ,  0.        ],
       [ 0.08722425, -0.09107554,  0.40173233, ...,  0.14676929,
        -0.19115317,  0.        ]], shape=(3067, 7615), dtype=float32)

In [12]:
pseudobulk.values

array([[-0.02277446, -0.17145205,  0.51165867, ..., -0.460747  ,
         0.09554577,  0.        ],
       [-0.01156116,  0.00919116, -0.16199684, ...,  0.27930033,
        -0.5089375 ,  0.        ],
       [ 0.03526187, -0.01786935, -0.11665964, ..., -0.18842447,
         0.27245724,  0.        ],
       ...,
       [ 0.10366416, -0.06074077,  0.11752832, ...,  0.17629874,
         0.06908941,  0.        ],
       [-0.34795213, -0.05310255, -0.09484243, ...,  0.1625731 ,
         0.3502773 ,  0.        ],
       [ 0.08722425, -0.09107554,  0.40173233, ...,  0.14676929,
        -0.19115317,  0.        ]], shape=(3067, 7615), dtype=float32)

In [13]:
pseudobulk.head()

,DPM1,SCYL3,C1orf112,FUCA2,GCLC,NIPAL3,LAS1L,ANKIB1,CYP51A1,KRIT1,...,ACACA,MRM1,H3C10,PAGR1,EXOC3L2,MSANTD7,SCO2,EEF1AKMT4,TBCE,ENSG00000284976
AACS,-0.022774,-0.171452,0.511659,-0.228417,0.619906,0.116226,-0.035529,-0.295850,-0.302337,-0.124856,...,-0.172017,-0.453360,-0.308899,-0.099776,-0.003454,-0.094202,0.318455,-0.460747,0.095546,0.0
AAK1,-0.011561,0.009191,-0.161997,-0.156069,-0.162580,0.116541,-0.200629,-0.067897,-0.106361,-0.288292,...,-0.050732,-0.226911,0.064994,-0.105179,0.086976,-0.209147,0.396256,0.279300,-0.508937,0.0
AAMDC,0.035262,-0.017869,-0.116660,-0.015759,0.273618,-0.094246,0.290642,0.238064,0.021766,0.056590,...,0.139434,-0.030634,0.045942,-0.017852,-0.003454,-0.053195,-0.304952,-0.188424,0.272457,0.0
AASDH,-0.141701,0.136460,-0.346613,-0.041003,0.091945,0.342165,0.072870,0.010665,-0.169431,-0.035681,...,-0.451282,0.196183,-0.032039,-0.137047,-0.003454,-0.135460,-0.223205,-0.278196,0.103782,0.0
ABCB6,-0.753226,0.138238,-0.314167,-0.122679,-0.135179,-0.130788,0.432411,-0.309368,-0.530626,-0.064554,...,-0.510939,-0.022511,0.154942,-0.050551,-0.003454,0.107164,-0.346343,0.079201,0.578457,0.0


In [14]:
pc = PCA(n_components=128).fit(pseudobulk.values)

emb = pc.components_.T

emb = np.pad(
            emb,
            ((0, 0), (0, 128 - emb.shape[1])),
            "constant",
            constant_values=(0),
        )
emb = pd.DataFrame(emb, index=pseudobulk.columns)

In [15]:
emb.shape

(7615, 128)

In [16]:
emb.head()

,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,127
DPM1,0.017852,-0.008386,0.004809,-0.008930,0.012819,0.006821,-0.004708,0.002799,-0.005158,-0.018293,...,0.004382,-0.009014,0.012763,0.013428,-0.005976,-0.003781,-0.000354,-0.007301,-0.005102,0.004310
SCYL3,0.005377,0.008444,-0.002147,0.005949,-0.004639,-0.002701,0.010599,-0.001798,0.003089,-0.003947,...,-0.000544,0.005936,0.002103,0.007449,0.005364,0.001521,-0.001183,0.017723,-0.006551,0.003120
C1orf112,0.009667,0.010345,-0.021137,0.008939,-0.010434,-0.005667,-0.006852,-0.003461,-0.007387,-0.014517,...,0.011712,-0.010785,0.018451,-0.009783,-0.002420,-0.009302,-0.008796,0.004426,0.006737,0.004358
FUCA2,0.011072,-0.004113,0.007188,0.002325,0.002379,-0.003468,-0.007696,0.010556,0.023638,-0.016024,...,-0.013605,0.023793,-0.018767,-0.007922,0.021208,-0.002787,-0.004148,-0.001333,-0.012292,-0.010492
GCLC,0.011877,0.015595,0.010133,-0.004411,-0.006267,0.004550,-0.003724,0.030798,0.012941,-0.010654,...,-0.010353,0.010251,0.003896,0.011509,-0.003214,-0.029736,-0.004768,0.007671,0.002891,0.032608


In [17]:
emb.index.name = None
emb.head()

,0,1,2,3,4,5,6,7,8,9,...,118,119,120,121,122,123,124,125,126,127
DPM1,0.017852,-0.008386,0.004809,-0.008930,0.012819,0.006821,-0.004708,0.002799,-0.005158,-0.018293,...,0.004382,-0.009014,0.012763,0.013428,-0.005976,-0.003781,-0.000354,-0.007301,-0.005102,0.004310
SCYL3,0.005377,0.008444,-0.002147,0.005949,-0.004639,-0.002701,0.010599,-0.001798,0.003089,-0.003947,...,-0.000544,0.005936,0.002103,0.007449,0.005364,0.001521,-0.001183,0.017723,-0.006551,0.003120
C1orf112,0.009667,0.010345,-0.021137,0.008939,-0.010434,-0.005667,-0.006852,-0.003461,-0.007387,-0.014517,...,0.011712,-0.010785,0.018451,-0.009783,-0.002420,-0.009302,-0.008796,0.004426,0.006737,0.004358
FUCA2,0.011072,-0.004113,0.007188,0.002325,0.002379,-0.003468,-0.007696,0.010556,0.023638,-0.016024,...,-0.013605,0.023793,-0.018767,-0.007922,0.021208,-0.002787,-0.004148,-0.001333,-0.012292,-0.010492
GCLC,0.011877,0.015595,0.010133,-0.004411,-0.006267,0.004550,-0.003724,0.030798,0.012941,-0.010654,...,-0.010353,0.010251,0.003896,0.011509,-0.003214,-0.029736,-0.004768,0.007671,0.002891,0.032608


In [18]:
with open("../cache/pathway_embeddings/HCT_presage_plus_prior.pkl", 'wb') as f:
    pickle.dump(emb, f)